# ESGReady — Topic 3 Lab: V1 Document Data Pipeline

**Project:** ESGReady — An Evidence-Grounded ESG Questionnaire Assistant for Malaysian SMEs  
**Scope in this notebook:** the C0 document-preparation component, initially for **K1 — SEDG Version 2 (July 2025)**. K1 is guidance, **not company evidence**. This is a Lab 3 data pipeline, not a trained AI assistant.

**Run record.** The saved Colab output records the 12 November 2026 run. This GitHub copy omits the upload widget and PDF previews. The record counts, checksums and repeatability results are retained.

**Before running:** Obtain K1 from the official link in `docs/source_register.md` under the permitted local-use rules; upload the PDF under the filename `SEDG-v2.pdf`. The source review in Topic 2 identifies eight candidate physical PDF pages (1-based): 26, 30, 34, 46, 53, 62, 65 and 75. Inspect extracted content against the PDF layout, particularly multi-column pages. A file's filename is not proof of its identity or correctness.

**Dependency:** `pypdf`. No language-model API key, model training or vector database required. This notebook implements Topic 3's Pathway A, and the stage sequence Cell 1 → upload → Cell 3 → A3/A4 → Cell 4 → 4A → 5A1 → 5A2 → B3 → B4.1 → 6A → 6B → Cell 7 → 8A → Cell 9.

### Part A — Step A2 / Cell 1: Initialise Colab workspace

In [1]:
from pathlib import Path
BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PIPE_DIR = BASE_DIR / 'data' / 'pipeline_v1'
DOC_DIR = BASE_DIR / 'docs'
for folder in (RAW_DIR, PROCESSED_DIR, PIPE_DIR, DOC_DIR):
    folder.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR; PROCESSED = PROCESSED_DIR; BASE = BASE_DIR; OUT = PROCESSED_DIR
print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)
print('Pipeline folder:', PIPE_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed
Pipeline folder: /content/week3_project/data/pipeline_v1


### Configuration and source acquisition (Step A1)

K1 is selected because **FR-03, FR-04, NFR-03, NFR-06 and RAI-06** require a versioned, citable, role-tagged guidance source. The full K1 PDF is retained locally but the **initial output covers only the eight candidate disclosure pages** from Lab 2. This is a bounded preparation choice, not a claim that all disclosures are correctly extracted.

K2 (NSRF FAQ) and K3 (OECD web article) remain separately identified in `docs/source_register.md`; they are **not silently mixed into K1's disclosure criteria**. To prepare K2 later, use a separately scoped PDF run with its selected, visually verified sections. K3 would require an HTML loader or conversion; it is outside this Pathway A run.

In [2]:
SOURCE = {
    'source_id': 'K1',
    'filename': 'SEDG-v2.pdf',
    'title': 'Simplified ESG Disclosure Guide for SMEs in Supply Chains',
    'publisher': 'Capital Markets Malaysia',
    'version': 'Version 2, July 2025 (from Topic 2 source register; reconfirm)',
    'url': 'https://sedg.capitalmarketsmalaysia.com/wp-content/uploads/2025/07/SEDG-v2.pdf',
    'role': 'disclosure_guidance',
    'pdf_pages_1based': [26, 30, 34, 46, 53, 62, 65, 75],
    'disclosure_by_page': {
        26: 'SEDG-E2.1', 30: 'SEDG-E3.1', 34: 'SEDG-E4.1',
        46: 'SEDG-S2.1', 53: 'SEDG-S4.1', 62: 'SEDG-G1.1',
        65: 'SEDG-G2.1', 75: 'SEDG-G4.1'
    },
}
CHUNK_SIZE = 500
CHOSEN_METHOD = 'paragraph-aware'  # Set to 'fixed' only after recording evidence.
PDF_SOURCE = RAW_DIR / SOURCE['filename']
print('Expected source:', PDF_SOURCE)
print('Selected physical PDF pages (1-based):', SOURCE['pdf_pages_1based'])

Expected source: /content/week3_project/data/raw/SEDG-v2.pdf
Selected physical PDF pages (1-based): [26, 30, 34, 46, 53, 62, 65, 75]


### Upload source file into Colab

Use this upload cell, or upload through the Colab Files sidebar into `/content/week3_project/data/raw`. **Run only in Colab.** It does not download the PDF automatically, preserving the source access decision. File bytes are not included in the GitHub starter package.

In [4]:
from google.colab import files
uploaded = files.upload()
for filename, file_bytes in uploaded.items():
    safe_name = Path(filename).name
    destination = RAW_DIR / safe_name
    destination.write_bytes(file_bytes)
    print('Uploaded:', destination, '| bytes:', destination.stat().st_size)
print('If necessary, rename the uploaded K1 PDF to SEDG-v2.pdf in the Files panel.')

Saving SEDG-v2.pdf to SEDG-v2.pdf
Uploaded: /content/week3_project/data/raw/SEDG-v2.pdf | bytes: 1928183
If necessary, rename the uploaded K1 PDF to SEDG-v2.pdf in the Files panel.


### Part A — Cell 3: Verify uploaded inputs

In [5]:
all_files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(all_files))
for item in all_files:
    print(item.relative_to(RAW_DIR), '| bytes:', item.stat().st_size)
assert all_files, 'Upload an approved project source into data/raw first.'
assert PDF_SOURCE.is_file(), f'Missing K1 PDF: {PDF_SOURCE}. Rename or update SOURCE["filename"].'
assert PDF_SOURCE.stat().st_size > 0, 'K1 PDF exists but is empty.'
print('PASS: intended source exists. Continue to actual content inspection.')

Input count: 1
SEDG-v2.pdf | bytes: 1928183
PASS: intended source exists. Continue to actual content inspection.


### Part A — Step A3: Inspect two representative pages

In [6]:
%pip -q install pypdf
from pypdf import PdfReader
reader = PdfReader(str(PDF_SOURCE))
print('Total physical PDF pages:', len(reader.pages))
assert max(SOURCE['pdf_pages_1based']) <= len(reader.pages), (
    'One of the Topic 2 page references is outside the uploaded PDF. '
    'Confirm source version and physical PDF page mapping.'
)
INSPECTION_PAGES = [26, 30]  # Selected Topic 2 page locations, not new observations.
inspection = []
for page_no in INSPECTION_PAGES:
    page_text = reader.pages[page_no - 1].extract_text() or ''
    record = {'source_id': SOURCE['source_id'], 'pdf_page_1based': page_no,
              'extracted_characters': len(page_text), 'has_text': bool(page_text.strip())}
    inspection.append(record)
    print('\nPhysical PDF page:', page_no,
          '| expected navigation ID:', SOURCE['disclosure_by_page'].get(page_no))
    print('Extracted characters:', len(page_text))
    print('Short inspection preview (check PDF visually):', repr(page_text[:160]))
assert all(x['has_text'] for x in inspection), 'Selected pages are blank/unreadable; inspect source.'
print('PASS: representative content extracted. VISUAL COLUMN/ID CHECK STILL REQUIRED.')

Total physical PDF pages: 82

Physical PDF page: 26 | expected navigation ID: SEDG-E2.1
Extracted characters: 1114

Physical PDF page: 30 | expected navigation ID: SEDG-E3.1
Extracted characters: 1123
PASS: representative content extracted. VISUAL COLUMN/ID CHECK STILL REQUIRED.


### Part A — Step A4: Draft inventory from observed file properties

This creates an **actual file-inspection record only after the source is uploaded**. Information about ownership/version comes from Topic 2 and remains labelled as such. It does **not** assert that redistribution is permitted.

In [7]:
from datetime import datetime, timezone
from hashlib import sha256
source_bytes = PDF_SOURCE.read_bytes()
source_digest = sha256(source_bytes).hexdigest()
run_date = datetime.now(timezone.utc).date().isoformat()
inventory_lines = [
    '# ESGReady — Week 3 Data Inventory (Colab observation)',
    '',
    f'Generated (UTC): {run_date}. Verify document identity and sharing rights separately.',
    '',
    '| Source | Filename / source | Type and purpose | Version / owner | Access / rights | Inspection evidence |',
    '| --- | --- | --- | --- | --- | --- |',
    f"| S01 / K1 | `{PDF_SOURCE.name}`; {SOURCE['url']} | PDF; `disclosure_guidance` | {SOURCE['version']}; {SOURCE['publisher']} | Public reference link; redistribution **not confirmed** | {len(reader.pages)} total physical pages, {PDF_SOURCE.stat().st_size} bytes; SHA-256 `{source_digest}` |",
]
for x in inspection:
    inventory_lines.append(
        f"| S01 subset p{x['pdf_page_1based']} | K1 physical PDF page {x['pdf_page_1based']} "
        f"| PDF page text sample | Same K1 snapshot | Same rights as K1 | "
        f"{x['extracted_characters']} extracted characters; readable={x['has_text']} |"
    )
inventory_lines += [
    '', '## Interpretation and limitations',
    '- These observations verify file loading and extraction, not correct column order or disclosure mapping.',
    '- K1 is disclosure guidance, not proof of company activity. No company records were processed.',
    '- K2 and K3 remain background sources listed in `docs/source_register.md`; this is a single-source V1 run.',
    '- Manually compare physical pages 26 and 30 with the visual PDF layout before signing off.',
]
inventory_path = DOC_DIR / 'week3_data_inventory.md'
inventory_path.write_text('\n'.join(inventory_lines) + '\n', encoding='utf-8')
print('Saved:', inventory_path)
print('Inspection subset count:', len(inspection))

Saved: /content/week3_project/docs/week3_data_inventory.md
Inspection subset count: 2


### Part B — Cell 4: Common metadata and provenance helpers

In [8]:
import hashlib, json
from datetime import datetime, timezone

def select_source(relative_path):
    source = RAW_DIR / relative_path
    if not source.is_file():
        raise FileNotFoundError(source)
    return source

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(entries):
    destination = BASE_DIR / 'data' / 'week3_manifest.json'
    payload = {
        'pipeline_version': 'week3-v1',
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'records': entries,
    }
    destination.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Saved Part B lineage:', destination)
    return payload

assert select_source(SOURCE['filename']) == PDF_SOURCE
print('Cell 4 ready: explicit selection + SHA-256 + manifest writer.')

Cell 4 ready: explicit selection + SHA-256 + manifest writer.


### Part B — Cell 4A: Initialise lineage registry

In [9]:
lineage_entries = {}
def register_lineage(source, output, processing):
    source, output = Path(source), Path(output)
    assert source.is_file(), f'Missing source: {source}'
    assert output.is_file() and output.stat().st_size > 0, f'Missing/empty output: {output}'
    assert str(processing).strip(), 'Explain the selected processing method.'
    lineage_entries[(str(source.resolve()), str(output.resolve()))] = {
        'source': source, 'output': output, 'processing': str(processing)
    }
    print('Registered:', source.name, '->', output.name)
print('Registry ready. Entries:', len(lineage_entries))

Registry ready. Entries: 0


### Part B — Cell 5A1: PDF page extraction, cleaning, method comparison

**Decision under test:** fixed 500-character chunks versus simple paragraph-aware 500-character chunks. Keep **physical PDF pages separate** so a chunk cannot silently cross a page boundary. Text extraction may still mix K1 columns; a manual check is mandatory. This is **not semantic chunking** and **not model inference**.

In [10]:
import re

def clean_pdf_page(text):
    # Conservative cleanup: normalize line endings and trailing whitespace.
    # Do not remove internal spaces which may represent multi-column layout.
    normalized = (text or '').replace('\r\n', '\n').replace('\r', '\n')
    return '\n'.join(line.rstrip() for line in normalized.split('\n')).strip()

def fixed_chunks(text, size=500):
    assert size > 0
    return [text[i:i+size] for i in range(0, len(text), size) if text[i:i+size].strip()]

def paragraph_chunks(text, size=500):
    assert size > 0
    result = []
    current = ''
    for para in re.split(r'\n\s*\n', text):
        para = para.strip()
        if not para:
            continue
        if len(para) > size:
            if current:
                result.append(current); current = ''
            result.extend(fixed_chunks(para, size))
        elif len(current) + len(para) + (2 if current else 0) <= size:
            current += ('\n\n' if current else '') + para
        else:
            if current: result.append(current)
            current = para
    if current: result.append(current)
    return result

def extract_selected_pages(path, selected_pages):
    pdf = PdfReader(str(path))
    if max(selected_pages) > len(pdf.pages):
        raise ValueError('Selected page indices exceed actual PDF page count.')
    page_rows = []
    for number in selected_pages:
        content = clean_pdf_page(pdf.pages[number - 1].extract_text() or '')
        if not content:
            raise ValueError(f'PDF page {number} contains no extractable text (OCR out of scope).')
        page_rows.append({'pdf_page_1based': number, 'text': content})
    return page_rows

selected_pages = extract_selected_pages(PDF_SOURCE, SOURCE['pdf_pages_1based'])
comparison = {'fixed': [], 'paragraph-aware': []}
for page in selected_pages:
    for label, method in (('fixed', fixed_chunks), ('paragraph-aware', paragraph_chunks)):
        chunks = method(page['text'], CHUNK_SIZE)
        comparison[label].append({'page': page['pdf_page_1based'],
                                  'chunk_count': len(chunks),
                                  'maximum_chunk_chars': max(map(len, chunks)),
                                  'preview': chunks[0][:120]})
print('Selected page count:', len(selected_pages))
for label, rows in comparison.items():
    print(label, '| chunks:', sum(x['chunk_count'] for x in rows),
          '| first chunk preview:', repr(rows[0]['preview']))
print('Compare boundaries and content with the PDF visually; record limitations in decisions.md.')

Selected page count: 8
fixed | chunks: 25
paragraph-aware | chunks: 26
Compare boundaries and content with the PDF visually; record limitations in decisions.md.


### Part B — Cell 5A2: Export the selected preparation method

The prepared records keep exact physical PDF page numbers, source ID, source version and the **source role**. `candidate_disclosure_id` is a Topic 2 navigation hint and **must not be treated as a verified per-chunk semantic label**.

In [11]:
assert CHOSEN_METHOD in {'fixed', 'paragraph-aware'}

def build_records(page_rows, source_file=PDF_SOURCE):
    chosen = fixed_chunks if CHOSEN_METHOD == 'fixed' else paragraph_chunks
    records = []
    digest = source_hash(source_file)
    for page in page_rows:
        p = page['pdf_page_1based']
        for index, chunk in enumerate(chosen(page['text'], CHUNK_SIZE), start=1):
            records.append({
                'record_id': f"K1-p{p:04d}-c{index:03d}",
                'source_id': 'K1', 'source_file': source_file.name,
                'source_sha256': digest,
                'source_role': SOURCE['role'],
                'source_version': SOURCE['version'],
                'source_url': SOURCE['url'],
                'pdf_page_1based': p,
                'printed_page_hint': p - 1,  # From Topic 2; visually confirm per page.
                'candidate_disclosure_id': SOURCE['disclosure_by_page'].get(p),
                'text': chunk,
                'processing': f'pdf-page-aware-{CHOSEN_METHOD}-char-{CHUNK_SIZE}',
            })
    return records

def export_jsonl(rows, output):
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('w', encoding='utf-8') as handle:
        for record in rows:
            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + '\n')

records = build_records(selected_pages)
prepared_path = PROCESSED_DIR / 'prepared_text.jsonl'
export_jsonl(records, prepared_path)
print('Prepared records:', len(records))
print('Prepared output:', prepared_path, '| bytes:', prepared_path.stat().st_size)
print('First record keys:', sorted(records[0]))

Prepared records: 26
Prepared output: /content/week3_project/data/processed/prepared_text.jsonl | bytes: 24313
First record keys: ['candidate_disclosure_id', 'pdf_page_1based', 'printed_page_hint', 'processing', 'record_id', 'source_file', 'source_id', 'source_role', 'source_sha256', 'source_url', 'source_version', 'text']


### Part B — Step B3: Validate the prepared export

In [12]:
assert prepared_path.is_file() and prepared_path.stat().st_size > 0
with prepared_path.open(encoding='utf-8') as handle:
    exported = [json.loads(line) for line in handle if line.strip()]
assert exported and len(exported) == len(records)
assert len({r['record_id'] for r in exported}) == len(exported), 'Record IDs are not unique.'
assert all(r['text'].strip() and len(r['text']) <= CHUNK_SIZE for r in exported)
assert all(r['source_role'] == 'disclosure_guidance' for r in exported)
assert all(r['pdf_page_1based'] in SOURCE['pdf_pages_1based'] for r in exported)
assert all(r['source_sha256'] == source_hash(PDF_SOURCE) for r in exported)
print('PASS: nonempty, well-formed, role-tagged, page-aware prepared export.')
print('Verified record count:', len(exported))
print('Representative PDF page:', exported[0]['pdf_page_1based'])
print('Inspect its column order and meaning against the original PDF before sign-off.')

PASS: nonempty, well-formed, role-tagged, page-aware prepared export.
Verified record count: 26
Representative PDF page: 26
Inspect its column order and meaning against the original PDF before sign-off.


### Part B — Step B4.1: Register source-to-output lineage

In [13]:
processing_description = (
    f"K1 selected physical pages={SOURCE['pdf_pages_1based']}; "
    f"conservative text cleanup; pdf-page-aware-{CHOSEN_METHOD}-char-{CHUNK_SIZE}; "
    'columns require manual inspection'
)
register_lineage(PDF_SOURCE, prepared_path, processing_description)

Registered: SEDG-v2.pdf -> prepared_text.jsonl


### Part B — Cell 6A: Create the common lineage manifest

In [14]:
lineage_records = []
assert lineage_entries, 'Register at least one real source-to-output relationship.'
for i, entry in enumerate(lineage_entries.values(), 1):
    src, out = entry['source'], entry['output']
    lineage_records.append({
        'source_id': f'S{i:02d}', 'project_source_id': SOURCE['source_id'],
        'source_filename': str(src.relative_to(RAW_DIR)),
        'source_sha256': source_hash(src),
        'source_role': SOURCE['role'],
        'source_version': SOURCE['version'],
        'source_url': SOURCE['url'],
        'access_rights': 'Public URL; redistribution not verified',
        'prepared_output': str(out.relative_to(BASE_DIR)),
        'prepared_sha256': source_hash(out),
        'processing_note': entry['processing'],
        'status': 'Prepared in this session; manual layout quality review pending',
    })
manifest = save_manifest(lineage_records)
print('Part B lineage entries:', len(manifest['records']))

Saved Part B lineage: /content/week3_project/data/week3_manifest.json
Part B lineage entries: 1


### Part B — Cell 6B: Verify recorded lineage

In [15]:
manifest_path = BASE_DIR / 'data' / 'week3_manifest.json'
saved_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert saved_manifest['records']
for entry in saved_manifest['records']:
    src = RAW_DIR / entry['source_filename']
    dst = BASE_DIR / entry['prepared_output']
    assert src.is_file() and dst.is_file()
    assert source_hash(src) == entry['source_sha256']
    assert source_hash(dst) == entry['prepared_sha256']
print('COMMON LINEAGE VALIDATION: PASS')

COMMON LINEAGE VALIDATION: PASS


### Part C — Cell 7: Common executable pipeline utility

A separate execution writes the Part C output under `data/pipeline_v1/text/`. The pipeline **reuses the same source and chosen transformation** as Part B. The manifest records a run timestamp, source/output hashes, record counts and source-to-output mapping. A fresh run should produce identical data bytes when inputs/configuration are unchanged.

In [16]:
PIPELINE_RUNS = {}

def run_esgready_text_pipeline():
    input_file = select_source(SOURCE['filename'])
    out_dir = PIPE_DIR / 'text'
    out_dir.mkdir(parents=True, exist_ok=True)
    page_rows = extract_selected_pages(input_file, SOURCE['pdf_pages_1based'])
    prepared_records = build_records(page_rows, input_file)
    if not prepared_records:
        raise ValueError('Pipeline created zero records.')
    output_file = out_dir / 'prepared_text.jsonl'
    export_jsonl(prepared_records, output_file)
    # Content and metadata stage contract.
    parsed_rows = [json.loads(line) for line in output_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    assert len(parsed_rows) == len(prepared_records) and all(r['text'].strip() for r in parsed_rows)
    assert len({r['record_id'] for r in parsed_rows}) == len(parsed_rows)
    mapping = [{'source': str(input_file.relative_to(BASE_DIR)),
                'output': str(output_file.relative_to(BASE_DIR))}]
    stable_run = {
        'source_sha256': source_hash(input_file),
        'prepared_sha256': source_hash(output_file),
        'record_count': len(prepared_records),
        'processed_pages_1based': SOURCE['pdf_pages_1based'],
        'source_output_mapping': mapping,
    }
    run_manifest = {
        'pipeline_version': 'week3-v1',
        'pipeline_kind': 'text',
        'executed_at_utc': datetime.now(timezone.utc).isoformat(),
        'source_role': SOURCE['role'],
        'source_version': SOURCE['version'],
        'source_url': SOURCE['url'],
        'method': f'pdf-page-aware-{CHOSEN_METHOD}-char-{CHUNK_SIZE}',
        'manual_layout_review': 'PENDING until compared with source PDF',
        'validation': 'PASS for presence, schema, page labels and content nonemptiness',
        **stable_run,
    }
    manifest_file = out_dir / 'pipeline_manifest.json'
    manifest_file.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    assert manifest_file.is_file() and manifest_file.stat().st_size > 0
    return stable_run
print('Cell 7 ready. No pipeline executed yet.')

Cell 7 ready. No pipeline executed yet.


### Part C — Cell 8A: Execute the selected document pipeline

In [17]:
first_run = run_esgready_text_pipeline()
PIPELINE_RUNS['text'] = first_run
print('Executed: text (PDF Pathway A)')
print('Records:', first_run['record_count'])
print('Source SHA-256:', first_run['source_sha256'])
print('Output SHA-256:', first_run['prepared_sha256'])
print('Part C output folder:', PIPE_DIR / 'text')

Executed: text (PDF Pathway A)
Records: 26
Source SHA-256: c7699997922c6735ceb1ae494eb2511841c66bd03a228807ae15eba3730ace19
Output SHA-256: 45f826424139dd72dbe2efa8f5de2505b88e7a31e33c53dbb642ec28ae5e7680
Part C output folder: /content/week3_project/data/pipeline_v1/text


### Part C — Cell 9: Repeatability and provenance verification

In [18]:
assert PIPELINE_RUNS, 'Run Cell 8A first.'
second_run = run_esgready_text_pipeline()
assert first_run['source_sha256'] == second_run['source_sha256'], 'Source changed.'
assert first_run['prepared_sha256'] == second_run['prepared_sha256'], 'Prepared output changed.'
assert first_run['record_count'] == second_run['record_count'], 'Record count changed.'
assert first_run['source_output_mapping'] == second_run['source_output_mapping'], 'Mapping changed.'
print('PASS: text — source fingerprints, prepared outputs and mappings match')
print('All selected pipelines passed repeatability smoke checks.')
print('Note: This is not a proof of layout quality, factual correctness or source redistribution rights.')

PASS: text — source fingerprints, prepared outputs and mappings match
All selected pipelines passed repeatability smoke checks.
Note: This is not a proof of layout quality, factual correctness or source redistribution rights.


### Save verified artefacts for your **existing** ESGReady repository

In Colab's Files sidebar, download and place the following into the matching paths of your local repository:  
- `/content/week3_project/docs/week3_data_inventory.md` → `docs/week3_data_inventory.md`
- `/content/week3_project/data/processed/prepared_text.jsonl` → `data/processed/` (only if sharing permitted)
- `/content/week3_project/data/week3_manifest.json` → `data/week3_manifest.json`
- `/content/week3_project/data/pipeline_v1/text/{prepared_text.jsonl,pipeline_manifest.json}` → `data/pipeline_v1/text/` (respect source rights)
- Save this **executed** notebook into `notebooks/week3_pipeline.ipynb`.

After running, update `docs/week3_pipeline_decisions.md` with observed counts, comparison, error notes and actual visual inspection results. Review `docs/week3_pipeline_diagram.png` against the executed stages, and mark its status as executed only after success. Add these files and the README Week 3 status to the same GitHub repository. Avoid uploading third-party PDF bytes or their full extracted text unless redistribution is authorised. No model-performance results are required for Topic 3.